In [ ]:
%reload_ext dotenv
%dotenv

In [ ]:
import json
from io import BytesIO

from PIL import Image
from google import genai
from google.genai import types

In [ ]:
def preview(img, size=600):
    p = img.copy()
    p.thumbnail((size, size))
    display(p)

In [ ]:
avatar = Image.open("../images/avatars/nimo.jpg")
wearable = Image.open("../images/garments/tops/sweater.jpg")

for im in [avatar, wearable]:
    preview(im, size=400)

In [ ]:
client = genai.Client()

prompt = """
Put this wearable on the avatar.
Preserve the avatar's Sims 3 / PS3 video game art style exactly.
Keep the same pose, background, and all other clothing unchanged.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents=[avatar, wearable, prompt],
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="3:4",
            image_size="512px",
        ),
    ),
)

assert response.parts is not None

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        genai_image = part.as_image()
        assert genai_image is not None
        assert genai_image.image_bytes is not None
        result = Image.open(BytesIO(genai_image.image_bytes))

usage = response.usage_metadata
assert usage is not None
print(
    f"\nToken usage: {usage.prompt_token_count} input, {usage.candidates_token_count} output, {usage.total_token_count} total"
)
print(f"Image size: {result.size} px")
preview(result)